In [0]:
# Cell 1 — Install libraries and download CMS policy PDFs
%pip install pymupdf sentence-transformers faiss-cpu

dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 137.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.4/426.4 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.6/444.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
# Cell 2 — Download real CMS PDFs from GitHub repo
import requests
import os
import tempfile

PDF_DIR = tempfile.mkdtemp()

# Exact filenames from GitHub repo
GITHUB_RAW = "https://raw.githubusercontent.com/Nupur-Gudigar/prior-auth-ai-agent/main/docs/cms_policies"

pdf_files = [
    "LCD - Lumbar MRI (L34220).pdf",
    "LCD - Diagnostic and Therapeutic Colonoscopy (L34213).pdf",
    "LCD - Major Joint Replacement (Hip and Knee) (L33618).pdf"
]

print("Downloading CMS policy PDFs from GitHub...")
for filename in pdf_files:
    # URL encode spaces and special characters
    encoded = filename.replace(" ", "%20").replace("(", "%28").replace(")", "%29")
    url = f"{GITHUB_RAW}/{encoded}"
    response = requests.get(url)
    if response.status_code == 200:
        filepath = os.path.join(PDF_DIR, filename)
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"✅ {filename} downloaded ({len(response.content):,} bytes)")
    else:
        print(f"❌ {filename} failed — status {response.status_code}")

print(f"\n🎉 CMS PDFs ready!")

✅ LCD - Lumbar MRI (L34220).pdf downloaded (331,389 bytes)
✅ LCD - Diagnostic and Therapeutic Colonoscopy (L34213).pdf downloaded (353,165 bytes)
✅ LCD - Major Joint Replacement (Hip and Knee) (L33618).pdf downloaded (268,519 bytes)

🎉 CMS PDFs ready!


In [0]:
# Cell 3 — Extract text from PDFs and chunk it
import fitz  # PyMuPDF
import os

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()
    return full_text

def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks = []
chunk_metadata = []

for filename in os.listdir(PDF_DIR):
    if filename.endswith(".pdf"):
        filepath = os.path.join(PDF_DIR, filename)
        print(f"Processing {filename}...")
        
        text = extract_text_from_pdf(filepath)
        print(f"  📄 Extracted {len(text):,} characters")
        
        chunks = chunk_text(text)
        print(f"  ✂️  Created {len(chunks)} chunks")
        
        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            chunk_metadata.append({
                "source": filename,
                "chunk_id": i
            })

print(f"\n✅ Total chunks ready for indexing: {len(all_chunks)}")

Processing LCD - Lumbar MRI (L34220).pdf...
  📄 Extracted 26,709 characters
  ✂️  Created 10 chunks
Processing LCD - Diagnostic and Therapeutic Colonoscopy (L34213).pdf...
  📄 Extracted 21,861 characters
  ✂️  Created 8 chunks
Processing LCD - Major Joint Replacement (Hip and Knee) (L33618).pdf...
  📄 Extracted 27,824 characters
  ✂️  Created 10 chunks

✅ Total chunks ready for indexing: 28


In [0]:
# Cell 4 — Build FAISS index and save permanently
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import io
import base64

# Load sentence transformer model
print("Loading sentence transformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded!")

# Convert chunks to vectors
print(f"\nConverting {len(all_chunks)} chunks to vectors...")
embeddings = model.encode(all_chunks, show_progress_bar=True)
print(f"✅ Embeddings shape: {embeddings.shape}")

# Build FAISS index
print("\nBuilding FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings, dtype=np.float32))
print(f"✅ FAISS index built — {index.ntotal} vectors indexed!")

# Save index as base64
faiss.write_index(index, "/tmp/policy_index.faiss")
with open("/tmp/policy_index.faiss", "rb") as f:
    index_bytes = base64.b64encode(f.read()).decode("utf-8")

# Save chunks to Delta table
chunks_data = [{
    "chunk_id": i,
    "chunk_text": all_chunks[i],
    "source": chunk_metadata[i]["source"]
} for i in range(len(all_chunks))]

chunks_df = spark.createDataFrame(pd.DataFrame(chunks_data))
chunks_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("policy_chunks")

# Save FAISS index to Delta table
index_df = spark.createDataFrame(
    pd.DataFrame([{"index_data": index_bytes}])
)
index_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("policy_faiss_index")

print("✅ policy_chunks saved to Delta!")
print("✅ policy_faiss_index saved to Delta!")
print("\n🎉 FAISS index permanently stored!")

Loading sentence transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded!

Converting 28 chunks to vectors...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings shape: (28, 384)

Building FAISS index...
✅ FAISS index built — 28 vectors indexed!
✅ policy_chunks saved to Delta!
✅ policy_faiss_index saved to Delta!

🎉 FAISS index permanently stored!


In [0]:
# Cell 5 — Test semantic search on real CMS policies
import faiss
import numpy as np
import base64
import pandas as pd

# Load FAISS index from Delta
print("Loading FAISS index from Delta...")
index_row = spark.table("policy_faiss_index").toPandas()
index_bytes = base64.b64decode(index_row["index_data"][0])
with open("/tmp/policy_index.faiss", "wb") as f:
    f.write(index_bytes)
index = faiss.read_index("/tmp/policy_index.faiss")
print("✅ FAISS index loaded!")

# Load chunks from Delta
chunks_df = spark.table("policy_chunks").toPandas()
all_chunks = chunks_df["chunk_text"].tolist()
chunk_sources = chunks_df["source"].tolist()
print(f"✅ {len(all_chunks)} chunks loaded!")

def search_policy(query, top_k=3):
    query_vector = model.encode([query])
    distances, indices = index.search(
        np.array(query_vector, dtype=np.float32), top_k
    )
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i+1,
            "source": chunk_sources[idx],
            "distance": distances[0][i],
            "text": all_chunks[idx][:300]
        })
    return results

# Test with real clinical queries
test_queries = [
    "MRI of the lumbar spine coverage criteria",
    "knee replacement medical necessity requirements",
    "colonoscopy screening age requirements"
]

for query in test_queries:
    print(f"\n🔍 Query: '{query}'")
    results = search_policy(query)
    for r in results:
        print(f"  Rank {r['rank']} | {r['source'][:40]} | distance: {r['distance']:.3f}")
        print(f"  → {r['text'][:200]}...")
    print()

Loading FAISS index from Delta...
✅ FAISS index loaded!
✅ 28 chunks loaded!

🔍 Query: 'MRI of the lumbar spine coverage criteria'
  Rank 1 | LCD - Lumbar MRI (L34220).pdf | distance: 0.515
  → flags"), MRI will be covered only if the patient has not responded to a reasonable trial of conservative management lasting at least four weeks. If a patient's limitations due to low back symptoms do ...
  Rank 2 | LCD - Lumbar MRI (L34220).pdf | distance: 0.563
  → lumbar MRI is sufficient to diagnose the patient's condition. However, a second lumbar MRI, for the same patient, may be allowed providing the documentation indicates that comparative test results wer...
  Rank 3 | LCD - Lumbar MRI (L34220).pdf | distance: 0.647
  → Differentiate solid from cystic tumors, Diagnose and localize spinal cord compression; Diagnose syringomyelia (progressive, chronic sensory disturbance, atrophy and spasticity of the spinal cord), dis...


🔍 Query: 'knee replacement medical necessity requirements'
  Rank 1